# Vocoder Comparison: HiFi-GAN vs BigVGAN v2

**Music Style Transfer Project** — Yotam & Gal

This notebook compares 3 vocoders on a real song using a Colab T4 GPU:
1. **HiFi-GAN** UNIVERSAL_V1 (14M params, speech-trained)
2. **BigVGAN v2** 22kHz (112M params, music-trained, drop-in upgrade)
3. **BigVGAN v2** 24kHz (112M params, highest fidelity)

### Setup
1. Set runtime to **GPU → T4** (Runtime → Change runtime type)
2. Upload your test WAV file to Google Drive: `My Drive/MusicProject_Colab/`
3. Run all cells in order

## 1. Check GPU

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("No GPU! Go to Runtime -> Change runtime type -> T4 GPU")

## 2. Clone Repository from GitHub

Since this is a **private repo**, you need a GitHub Personal Access Token (PAT).

**One-time setup** — create a PAT:
1. Go to https://github.com/settings/tokens
2. Click "Generate new token (classic)"
3. Check the **repo** scope → Generate
4. Copy the token and paste it below when prompted

*The token is entered securely (hidden) and never saved.*

In [ ]:
import os
from getpass import getpass

REPO = "galgeva1/MusicProject"
PROJECT_DIR = "/content/MusicProject"

if os.path.isdir(PROJECT_DIR):
    print(f"Repo already cloned at {PROJECT_DIR}")
else:
    token = getpass("GitHub Personal Access Token: ")
    !git clone https://{token}@github.com/{REPO}.git {PROJECT_DIR}
    print(f"\nCloned to {PROJECT_DIR}")

# Show project structure
for root, dirs, files in os.walk(PROJECT_DIR):
    dirs[:] = [d for d in dirs if d not in (".git", "__pycache__")]
    level = root.replace(PROJECT_DIR, '').count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        size = os.path.getsize(os.path.join(root, f))
        unit = "MB" if size > 1e6 else "KB"
        val = size/1e6 if size > 1e6 else size/1e3
        print(f"{indent}  {f} ({val:.1f} {unit})")

## 3. Mount Google Drive & Copy Test WAV

Upload your test WAV to: **My Drive → MusicProject_Colab/**

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, shutil
from pathlib import Path

DRIVE_FOLDER = "/content/drive/MyDrive/MusicProject_Colab"
PROJECT_DIR = Path("/content/MusicProject")

if not os.path.isdir(DRIVE_FOLDER):
    os.makedirs(DRIVE_FOLDER, exist_ok=True)
    print(f"Created {DRIVE_FOLDER} — upload your WAV there and re-run this cell")
else:
    wav_files = list(Path(DRIVE_FOLDER).glob("*.wav"))
    if wav_files:
        source_wav = wav_files[0]
        song_name = source_wav.stem
        test_wav = PROJECT_DIR / f"{song_name}.wav"
        if not test_wav.exists():
            shutil.copy2(str(source_wav), str(test_wav))
        print(f"Test WAV: {song_name} ({test_wav.stat().st_size/1e6:.1f} MB)")
    else:
        print(f"No WAV files found in {DRIVE_FOLDER}")
        print("Upload a WAV file there and re-run this cell")
        song_name = None
        test_wav = None

## 4. Verify HiFi-GAN Checkpoint (from repo)

In [ ]:
from pathlib import Path
PROJECT_DIR = Path("/content/MusicProject")
ckpt_dir = PROJECT_DIR / "postprocessing" / "hifigan_checkpoints" / "UNIVERSAL_V1"

print("HiFi-GAN checkpoint files:")
for fname in ["config.json", "g_02500000"]:
    fp = ckpt_dir / fname
    if fp.exists():
        print(f"  ✓ {fname} ({fp.stat().st_size/1e6:.1f} MB)")
    else:
        print(f"  ✗ {fname} — MISSING!")
        print(f"    Expected at: {ckpt_dir}")

## 5. Install Dependencies

In [ ]:
%%capture install_output
# Install BigVGAN + pin huggingface_hub (compatibility fix)
!pip install bigvgan "huggingface_hub<1.0" soundfile librosa

In [ ]:
import bigvgan, soundfile, librosa, torch
print(f"bigvgan OK")
print(f"PyTorch {torch.__version__}, CUDA {torch.cuda.is_available()}")
print(f"librosa {librosa.__version__}")

## 6. Run All 3 Vocoders

Each vocoder:
- Loads audio → computes mel spectrogram (using its own native format)
- Runs mel through the generator → produces reconstructed waveform
- Saves output WAV

In [ ]:
import sys, time, os, torch
PROJECT_DIR = "/content/MusicProject"
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

from postprocessing.vocoder_factory import create_vocoder, AVAILABLE_VOCODERS

input_path = str(test_wav)
results = {}

for voc_name, voc_info in AVAILABLE_VOCODERS.items():
    print("=" * 70)
    print(f"VOCODER: {voc_name} — {voc_info['description']}")
    print("=" * 70)

    output_path = f"{PROJECT_DIR}/{song_name}_{voc_name}_reconstructed.wav"

    # Load model
    t0 = time.time()
    vocoder = create_vocoder(voc_name, device="cuda")
    load_time = time.time() - t0

    # Run inference
    t0 = time.time()
    vocoder.wav_to_wav(input_path, output_path)
    infer_time = time.time() - t0

    file_size = os.path.getsize(output_path) / 1e6
    results[voc_name] = {
        "load_time": load_time,
        "infer_time": infer_time,
        "file_size_mb": file_size,
        "output_path": output_path
    }
    print(f"  Load: {load_time:.1f}s | Inference: {infer_time:.1f}s | Output: {file_size:.1f} MB")

    # Free GPU memory
    del vocoder
    torch.cuda.empty_cache()
    print()

## 7. Results Comparison

In [ ]:
import librosa, numpy as np, os
from postprocessing.vocoder_factory import AVAILABLE_VOCODERS

print(f"Song: {song_name}")
print(f"Input: {test_wav}\n")

header = f"{"Vocoder":<15} {"Load (s)":<10} {"Infer (s)":<10} {"Size (MB)":<10} {"RMS":<8}"
print(header)
print("-" * len(header))

# Original stats
y_orig, _ = librosa.load(input_path, sr=22050, mono=True)
rms_orig = np.sqrt(np.mean(y_orig**2))
orig_size = os.path.getsize(input_path)/1e6
print(f"{"ORIGINAL":<15} {"-":<10} {"-":<10} {orig_size:<10.1f} {rms_orig:<8.4f}")

# Vocoder stats
for voc_name, r in results.items():
    sr_voc = AVAILABLE_VOCODERS[voc_name]["sample_rate"]
    y_r, _ = librosa.load(r["output_path"], sr=sr_voc, mono=True)
    rms = np.sqrt(np.mean(y_r**2))
    print(f"{voc_name:<15} {r["load_time"]:<10.1f} {r["infer_time"]:<10.1f} {r["file_size_mb"]:<10.1f} {rms:<8.4f}")

## 8. Listen & Compare

In [ ]:
from IPython.display import Audio, display, HTML
import librosa
from postprocessing.vocoder_factory import AVAILABLE_VOCODERS

display(HTML("<h3>Original</h3>"))
y_orig, _ = librosa.load(input_path, sr=22050, mono=True)
display(Audio(y_orig, rate=22050))

for voc_name, r in results.items():
    sr_voc = AVAILABLE_VOCODERS[voc_name]["sample_rate"]
    desc = AVAILABLE_VOCODERS[voc_name]["description"]
    display(HTML(f"<h3>{voc_name} — {desc}</h3>"))
    y, _ = librosa.load(r["output_path"], sr=sr_voc, mono=True)
    display(Audio(y, rate=sr_voc))

## 9. Save Outputs to Google Drive

In [ ]:
import shutil
from pathlib import Path

DRIVE_FOLDER = "/content/drive/MyDrive/MusicProject_Colab"
output_dir = Path(DRIVE_FOLDER) / "vocoder_outputs"
output_dir.mkdir(exist_ok=True)

for voc_name, r in results.items():
    dst = output_dir / Path(r["output_path"]).name
    shutil.copy2(r["output_path"], str(dst))
    print(f"  Saved: {dst.name}")

print(f"\nAll outputs saved to: {output_dir}")
print("You can find them in Google Drive -> MusicProject_Colab -> vocoder_outputs/")